# 03 Feature Engineering

This notebook documents the current engineered features and feature-set policy.

## Reproducibility note

These notebooks are lightweight narrative companions to the source-controlled pipeline. The canonical workflow lives in src, generated metrics and figures live under reports, and the final selected model is models/xgboost_public.pkl. The protected attribute SEX is excluded from active model training and retained only for fairness diagnostics.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, Markdown

from src.utils import ROOT_DIR, REPORTS_DIR, MODELS_DIR, load_dataset_auto, load_model
from src.data_preprocessing import TARGET_COL

ROOT = ROOT_DIR
REPORTS = REPORTS_DIR
MODELS = MODELS_DIR
PROCESSED_DATA = ROOT / 'data' / 'processed' / 'uci_taiwan_credit_default_processed.csv'

pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid')


def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def show_image_if_exists(path: Path, width: int = 900):
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing optional image: {path.relative_to(ROOT) if path.is_absolute() else path}')


def load_project_dataset():
    try:
        frame, source = load_dataset_auto()
        return frame, source
    except Exception as exc:
        if PROCESSED_DATA.exists():
            print(f'UCI loader failed; using processed local fallback. Reason: {exc}')
            return pd.read_csv(PROCESSED_DATA), PROCESSED_DATA
        raise

raw_df, DATA_SOURCE = load_project_dataset()
print('Project root:', ROOT)
print('Data source:', DATA_SOURCE)
print('Rows:', f'{len(raw_df):,}')
print('Columns:', raw_df.shape[1])

from src.data_preprocessing import (
    APPLICATION_TIME_FEATURES,
    BEHAVIORAL_MONITORING_ONLY_FEATURES,
    FEATURE_POLICY_NOTE,
    FEATURE_SET_APPLICATION,
    FEATURE_SET_FULL_DIAGNOSTIC,
    UCI_TIMING_NOTE,
    get_feature_columns,
    prepare_modeling_table,
)

prepared = prepare_modeling_table(raw_df, target_col=TARGET_COL)
application_features = get_feature_columns(prepared, FEATURE_SET_APPLICATION)
full_diagnostic_features = get_feature_columns(prepared, FEATURE_SET_FULL_DIAGNOSTIC)
print('Application feature count:', len(application_features))
print('Full diagnostic feature count:', len(full_diagnostic_features))

Project root: D:\PGDBA\Projects\Credit Default Risk\credit-default-xai
Data source: uci:\default_credit_card
Rows: 30,000
Columns: 38


Application feature count: 36
Full diagnostic feature count: 37


In [2]:
display(pd.DataFrame({'application_public_features': pd.Series(application_features)}))
display(pd.DataFrame({'behavioral_monitoring_only_features': pd.Series(BEHAVIORAL_MONITORING_ONLY_FEATURES)}))
print(FEATURE_POLICY_NOTE)
print(UCI_TIMING_NOTE)

,application_public_features
0,LIMIT_BAL
1,EDUCATION
2,MARRIAGE
3,AGE
4,PAY_0
5,PAY_2
6,PAY_3
7,PAY_4
8,PAY_5
9,PAY_6


,behavioral_monitoring_only_features


application_public excludes the direct protected attribute SEX from active training features while retaining SEX in the audit dataframe for fairness analysis.
PAY_0 to PAY_6 are historical repayment-status variables available before the next-month default target and are not treated as leakage for this modeling question.


## Final framing

The final model uses the application_public feature set. SEX is excluded from the model inputs and retained in audit data for protected-attribute analysis.